<a href="https://colab.research.google.com/github/osleysorio/itacedemy_repo1_osley/blob/main/sprint%207/ejercicio2_n3_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Algoritmo  y promgramas
nivel 3
Ejercicio 2: Procesamiento automático de datos deportivos

### ==========================================
### 1. IMPORTAR LIBRERÍAS
### ==========================================

In [ ]:
import re
import pandas as pd
from google.colab import drive

### ==========================================
### 2. DEFINICIÓN DE FUNCIONES
### ==========================================

## Monta Google Drive en el entorno de Colab.

####Preparación del entorno Conectar el sistema con el almacenamiento en la nube (Google Drive) para poder acceder a los archivos.

In [ ]:
def conectar_drive():
    drive.mount('/content/drive')


##Lee un archivo de texto y devuelve una lista con sus líneas.

####Localizar y abrir el archivo de texto que contiene el historial de los partidos de fútbol.
#####Leer todo el documento y separar el texto línea por línea, guardándolo en una lista de notas.

In [ ]:
def leer_lineas_archivo(ruta):
    with open(ruta, 'r', encoding='utf-8') as f:
        return f.readlines()


####Por cada línea de texto leída en el archivo, realizar lo siguiente:
####Borrar los espacios en blanco vacíos que queden al inicio o al final de la frase.
#### Si la línea está completamente en blanco, ignorarla y pasar a la siguiente.
####Buscar dentro de la frase un patrón numérico separado por un guion (ejemplo: Número-Número).
####Si no se encuentra este patrón, descartar la línea por estar defectuosa.Si se encuentra, guardar el primer número como "Goles del Local" y el segundo como "Goles del Visitante".
####Cortar la frase en dos partes usando el guion (-) como separador, la parte izquierda será para el equipo local y la parte derecha para el visitante.
#### Limpiar los textos de ambas partes: borrar cualquier número restante, dos puntos, o paréntesis para que queden únicamente las letras de los nombres de los clubes.
####Guardar el nombre limpio del "Equipo Local" y del "Equipo Visitante" junto a sus respectivos goles en una ficha técnica del partido.
  

In [ ]:
def extraer_datos_partido(linea):

    linea = linea.strip()
    if not linea:
        return None

    # Paso 1: Extraer los goles antes de limpiar el texto
    score_match = re.search(r'(\d+)-(\d+)', linea)
    if not score_match:
        return None

    home_goals = int(score_match.group(1))
    visitor_goals = int(score_match.group(2))

    # Paso 2: Utilizar el guion '-' como separador definitivo para dividir la línea en dos partes
    partes_linea = linea.split('-', maxsplit=1)
    if len(partes_linea) != 2:
        return None

    # Paso 3: Eliminar números y caracteres especiales para aislar nombres
    home_club = re.sub(r'[\d+:()]+', '', partes_linea[0]).strip()
    visitor_club = re.sub(r'[\d+:()]+', '', partes_linea[1]).strip()

    if home_club and visitor_club:
        return {
            'home_club': home_club,
            'home_goals': home_goals,
            'visitor_goals': visitor_goals,
            'visitor_club': visitor_club
        }

    return None


In [ ]:
def procesar_datos_deportivos(ruta_archivo):
    """Coordina la lectura, el filtrado línea por línea y crea la tabla de partidos básica."""
    lineas = leer_lineas_archivo(ruta_archivo)
    datos_extraidos = []

    for linea in lineas:
        partido_datos = extraer_datos_partido(linea)
        if partido_datos:
            datos_extraidos.append(partido_datos)

    return pd.DataFrame(datos_extraidos)


   ## Genera una única tabla de posiciones unificada con las columnas ordenadas según el requerimiento: goles_favor, goles_contra, ganados, empatados, perdidos, puntos.


  Tomar la pizarra con todos los equipos y ordenarla de arriba hacia abajo, colocando primero al equipo que acumuló más Puntos y al último al que obtuvo menos.Crear una columna llamada "Orden" y asignar los números del 1 en adelante (1º, 2º, 3º...) según la posición que ocuparon tras el ordenamiento.Buscar en la columna de Goles a Favor cuál es la cifra más alta de toda la tabla y apartar el nombre de ese equipo como el "Más Goleador".Buscar en la columna de Goles en Contra cuál es la cifra más alta de toda la tabla y apartar el nombre de ese equipo como el "Más Goleado".

In [ ]:
def generar_tabla_posiciones_completa(df_partidos):

    resultados = {}

    for index, row in df_partidos.iterrows():
        home = row['home_club']
        visitor = row['visitor_club']
        home_g = row['home_goals']
        visitor_g = row['visitor_goals']

        # Inicializar estructuras si el equipo no ha sido registrado
        for equipo in [home, visitor]:
            if equipo not in resultados:
                resultados[equipo] = {
                    'goles_favor': 0, 'goles_contra': 0,
                    'ganados': 0, 'empatados': 0, 'perdidos': 0,
                    'puntos': 0
                }

        # Acumular goles a favor y en contra
        resultados[home]['goles_favor'] += home_g
        resultados[home]['goles_contra'] += visitor_g
        resultados[visitor]['goles_favor'] += visitor_g
        resultados[visitor]['goles_contra'] += home_g

        # Calcular resultados del partido y asignar puntos
        if home_g > visitor_g:
            resultados[home]['ganados'] += 1
            resultados[home]['puntos'] += 3
            resultados[visitor]['perdidos'] += 1
        elif visitor_g > home_g:
            resultados[visitor]['ganados'] += 1
            resultados[visitor]['puntos'] += 3
            resultados[home]['perdidos'] += 1
        else:
            resultados[home]['empatados'] += 1
            resultados[home]['puntos'] += 1
            resultados[visitor]['empatados'] += 1
            resultados[visitor]['puntos'] += 1

    # Convertir el diccionario a DataFrame
    df_tabla = pd.DataFrame.from_dict(resultados, orient='index')
    df_tabla.index.name = 'equipo'
    df_tabla = df_tabla.reset_index()

    # Ordenar las filas estrictamente por los puntos acumulados de mayor a menor
    df_tabla = df_tabla.sort_values(by='puntos', ascending=False).reset_index(drop=True)

    # Reordenar las columnas exactamente con el orden solicitado por el usuario
    columnas_ordenadas = ['equipo', 'goles_favor', 'goles_contra', 'ganados', 'empatados', 'perdidos', 'puntos']
    return df_tabla[columnas_ordenadas]

###    Identifica y devuelve el equipo que ha marcado la mayor cantidad de goles a favor.

    Args:
        df_clasificacion_final (pd.DataFrame): DataFrame que contiene los datos
                                               de clasificación, incluyendo 'equipo' y 'goles_favor'.

    Returns:
        pd.Series: Una serie que contiene el nombre del equipo y su total de goles a favor.


In [ ]:
def obtener_equipo_mas_goleador(df_clasificacion_final):

    if df_clasificacion_final.empty:
        return "No hay datos de clasificación para determinar el equipo más goleador."

    equipo_mas_goleador = df_clasificacion_final.loc[df_clasificacion_final['goles_favor'].idxmax()]
    return equipo_mas_goleador[['equipo', 'goles_favor']]


###    Identifica y devuelve el equipo que ha recibido la mayor cantidad de goles en contra.

    Args:
        df_clasificacion_final (pd.DataFrame): DataFrame que contiene los datos
                                               de clasificación, incluyendo 'equipo' y 'goles_contra'.

    Returns:
        pd.Series: Una serie que contiene el nombre del equipo y su total de goles en contra.
    """

In [ ]:
def obtener_equipo_mas_goleador(df_clasificacion_final):

    if df_clasificacion_final.empty:
        return "No hay datos de clasificación para determinar el equipo más goleado."

    equipo_mas_goleador = df_clasificacion_final.loc[df_clasificacion_final['goles_contra'].idxmax()]
    return equipo_mas_goleador[['equipo', 'goles_contra']]


### ==========================================
### 3. EJECUCIÓN DEL PROGRAMA
### ==========================================

####Imprimir en pantalla la Tabla de Posiciones Histórica completa con todas las columnas estadísticas ordenadas.
####Mostrar un letrero con el nombre y los goles del Equipo Más Goleador.
####Mostrar un letrero con el nombre y los goles del Equipo Más Goleado.
####Imprimir la lista final simplificada mostrando únicamente el número de puesto (Orden) y el nombre de cada equipo.

In [ ]:
# ==========================================
# 3. EJECUCIÓN DEL PROGRAMA
# ==========================================

# Conectar almacenamiento
conectar_drive()

# Definir ruta y procesar origen de datos
ruta_historico = '/content/drive/MyDrive/Colab Notebooks/historic partits.txt'
df_partidos = procesar_datos_deportivos(ruta_historico)

# Generar la tabla de clasificación final
df_clasificacion_final = generar_tabla_posiciones_completa(df_partidos)

# Generar tabla con número de orden (Movido aquí para evitar NameError)
df_clasificacion_final['Orden'] = df_clasificacion_final.index + 1
df_clasificacion_con_orden = df_clasificacion_final[['Orden', 'equipo']]

# Desplegar resultados
print("\n--- TABLA DE POSICIONES HISTÓRICA ---")
display(df_clasificacion_final)

print("\n--- Equipo Más Goleador ---")
equipo_goleador = obtener_equipo_mas_goleador(df_clasificacion_final)
print(equipo_goleador)

print("\n--- Equipo Más Goleado (Más Goles en Contra) ---")
equipo_goleado_contra = obtener_equipo_mas_goleado(df_clasificacion_final)
print(equipo_goleado_contra)

print("\n--- TABLA DE CLASIFICACIÓN CON NÚMERO DE ORDEN ---")
print(df_clasificacion_con_orden.to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- TABLA DE POSICIONES HISTÓRICA ---


,equipo,goles_favor,goles_contra,ganados,empatados,perdidos,puntos,Orden
0,Girona FC,139,94,31,3,13,96,1
1,Llagostera,159,142,29,7,20,94,2
2,Sabadell,141,121,26,7,15,85,3
3,Cornellà,147,146,25,7,22,82,4
4,RCD Espanyol,131,144,23,11,21,80,5
5,Figueres,161,154,23,10,23,79,6
6,Lleida Esportiu,129,133,23,6,23,75,7
7,Terrassa,147,164,20,14,23,74,8
8,FC Barcelona,125,115,22,7,15,73,9
9,Vilafranca,157,172,20,12,25,72,10



--- Equipo Más Goleador ---
equipo          Vilafranca
goles_contra           172
Name: 9, dtype: object

--- Equipo Más Goleado (Más Goles en Contra) ---
equipo          Vilafranca
goles_contra           172
Name: 9, dtype: object

--- TABLA DE CLASIFICACIÓN CON NÚMERO DE ORDEN ---
 Orden              equipo
     1           Girona FC
     2          Llagostera
     3            Sabadell
     4            Cornellà
     5        RCD Espanyol
     6            Figueres
     7     Lleida Esportiu
     8            Terrassa
     9        FC Barcelona
    10          Vilafranca
    11            Badalona
    12 Nàstic de Tarragona
    13       Reus Deportiu
    14          Granollers
    15                Olot
    16         Sant Andreu
    17             Manlleu
    18          Cerdanyola
    19                Prat
    20              Europa
